In [ ]:
from pyspark.errors import PySparkException
from py4j.protocol import Py4JJavaError
from pyspark.sql.functions import current_timestamp

In [ ]:
# Usuário, senha, host e port do banco de dados
usuario = dbutils.secrets.get("databases_lh_nautical", "sql_server_user")
senha = dbutils.secrets.get("databases_lh_nautical", "sql_server_password")
host = dbutils.secrets.get("databases_lh_nautical", "sql_server_host")
port = dbutils.secrets.get("databases_lh_nautical", "sql_server_port")

In [ ]:
def consulta_tabela_jdbc(jdbc_url, propriedades_conexao, nome_tabela_completa, numero_particao, coluna_referencia_paralelizacao = None):

    if coluna_referencia_paralelizacao is not None:
        limites_df = spark.read.jdbc(
            url = jdbc_url,
            table = f"(SELECT MIN({coluna_referencia_paralelizacao}) AS min_atributo, MAX({coluna_referencia_paralelizacao}) AS max_atributo FROM {nome_tabela_completa}) AS bounds",
            properties = propriedades_conexao
        )

        limites = limites_df.collect()[0]
        limite_inferior = limites['min_atributo']
        limite_superior = limites['max_atributo']

        df_temp = spark.read.jdbc(
            url = jdbc_url, 
            table = nome_tabela_completa,
            column = coluna_referencia_paralelizacao,
            lowerBound = limite_inferior if limite_inferior is not None else None,
            upperBound = limite_superior if limite_superior is not None else None,
            numPartitions = numero_particao, 
            properties = propriedades_conexao
        )
    else:
        df_temp = spark.read.jdbc(
            url = jdbc_url, 
            table = nome_tabela_completa,
            properties = propriedades_conexao
        )
    
    return df_temp

In [ ]:
# Info do banco de dados
jdbc_url = f"jdbc:sqlserver://{host}:{port};databaseName=lh_nautical;encrypt=false;trustServerCertificate=true;"
propriedades_conexao = {
    "user": usuario,
    "password": senha,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

# Info do catalog do desenvolvedor
catalog_name = dbutils.widgets.get("catalog_name")
schema = dbutils.widgets.get("schema")
numero_particao = dbutils.widgets.get("numPartitions")

In [ ]:
# Query para descobrir todas as tabelas do esquema 'dbo'
query_tabelas = "(select table_name from information_schema.tables where table_schema = 'dbo' and table_type = 'Base table') as tabelas"

try:
  # Tentaiva de conexão com o banco para obter as lista de tabelas para um DataFrame e convertemos para uma lista em Python
  df_tabelas = consulta_tabela_jdbc(jdbc_url, propriedades_conexao, query_tabelas, numero_particao)

  tabelas = df_tabelas.collect() 
  lista_tabelas = [row['table_name'] for row in df_tabelas.collect()]

except Py4JJavaError as e:
  erro_msg = str(e)
  print("[Erro de conexão/JDBC] - Falha ao conectar ou executar a query no banco de dados." )
  print(f"Detalhes do erro: {erro_msg}")
  lista_tabelas = []
  raise erro_msg
  
except PySparkException as e:
  erro_msg = str(e)
  print(f"[Erro do Spark] - Ocorreu uma falha interna no Spark: {erro_msg}")
  lista_tabelas = []
  raise erro_msg

except Exception as e:
  erro_msg = str(e)
  print(f"[Erro Inesperado] - {erro_msg}")
  lista_tabelas = []
  raise erro_msg


In [ ]:
# Ingestão de dados
tabelas_com_erro = []

for nome_tabela in lista_tabelas:
    print(f"Importanto a tabela: {nome_tabela}...")
    nome_tabela_completa = f"dbo.{nome_tabela}"

    try:
        if nome_tabela not in ['variant_attribute_values', 'product_suppliers', 'stock_levels']:
            df_temp = consulta_tabela_jdbc(jdbc_url, propriedades_conexao, nome_tabela_completa, numero_particao, "id")
        
        elif nome_tabela in ['product_suppliers', 'stock_levels']:
            df_temp = consulta_tabela_jdbc(jdbc_url, propriedades_conexao, nome_tabela_completa, numero_particao, "updated_at")

        else:
            df_temp = consulta_tabela_jdbc(jdbc_url, propriedades_conexao, nome_tabela_completa, numero_particao)

        # Adiciona a coluna com a data e hora atual do Spark
        # Caso as tabelas já estejam importadas sem o current_timestamp ou alguma outra coluna tem-se duas opções:
        # 1. Criar a(s) coluna(s) nas tabelas usando SQL
        # 2. Usar o mergeSchema 
        df_temp_com_data = df_temp.withColumn("data_ingestao", current_timestamp())

        df_temp_com_data.write.format("delta").mode("append").saveAsTable(f"{catalog_name}.{schema}.{nome_tabela}")
        print(f"Tabela {nome_tabela} integrada com sucesso!")

    except Py4JJavaError as e:
        erro_msg = str(e)
        print("[Erro de conexão/JDBC] - Falha ao conectar ou executar a query no banco de dados." )
        print(f"Erro ao processar tabela {nome_tabela}: {erro_msg}")
        # Registra a tabela que falhou para o resumo final
        tabelas_com_erro.append(nome_tabela)
        
        # Dispara o e-mail de alerta
        raise erro_msg
    
    except PySparkException as e:
        erro_msg = str(e)
        print(f"[Erro do Spark] - Ocorreu uma falha interna no Spark: {erro_msg}")
        print(f"Erro ao processar tabela {nome_tabela}: {erro_msg}")

        # Registra a tabela que falhou para o resumo final
        tabelas_com_erro.append(nome_tabela)

        # Dispara o e-mail de alerta
        raise erro_msg
    
    except Exception as e:
        erro_msg = str(e)
        print(f"[Erro Inesperado] - {erro_msg}")

        # Registra a tabela que falhou para o resumo final
        tabelas_com_erro.append(nome_tabela)

        # Dispara o e-mail de alerta
        raise erro_msg


print("Ingestão finalizada!")
if tabelas_com_erro:
    print(f"Atenção! As seguintes tabelas falharam e geraram alertas: {tabelas_com_erro}")
else:
    print("Sucesso absoluto! Todas as tabelas foram copiadas sem erros.")